# Tide-Table — a quantitative teardown 🔬
### CAPE vs forward returns · 10y vs 1y R² · the valuation ladder · the implied-return fit

![Signal: Real](https://img.shields.io/badge/Signal-Real-2ea44f?style=flat-square)
![Tradability: Fragile](https://img.shields.io/badge/Tradability-Fragile-dab617?style=flat-square)
![Forecasts long-run returns?: Confirmed](https://img.shields.io/badge/Forecasts_long--run_returns%3F-Confirmed-8b949e?style=flat-square)

The deep companion to the [notebook for the curious](01_for_the_curious.ipynb). We quantify CAPE's long-horizon forecasting power and its short-horizon limit.

> ⚠️ **Not investment advice.** Shiller CAPE + forward real total returns (datahub mirror), 1900–2016. Sources in [`docs/references.md`](../docs/references.md).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))  # repo root (quantlab/)
sys.path.insert(0, os.path.abspath(".."))        # study package (tide_table/)
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 5.5); plt.rcParams["axes.grid"] = True
import numpy as np, pandas as pd
from tide_table import data, strategy as st
d = data.fetch_shiller(start_year=1900)        # cache-first
ten = st.forward_corr(d["cape"], d["fwd10"]); one = st.forward_corr(d["cape"], d["fwd1"])


## Verdict, up front

| Axis | Stamp | Why |
|---|---|---|
| Signal | **Real** | corr −0.51, R² 0.28 at 10y; monotone ladder |
| Tradability | **Fragile** | 1y R² 0.05 — not a timer |
| Forecasts long-run? | **Confirmed** | valuation carries it (vs Fed Model) |

> 💡 *In plain words:* the tide is forecastable; the next wave is not.

## 1 · The claim, steelmanned

- **H₁:** CAPE forecasts the next-10y real return (negative, significant).
- **H₂:** the cheap/expensive bucket ordering is monotone.
- **H₃ (the limit):** the 1-year relationship is far weaker — CAPE is not a timer.

## 2 · So what? — what rides on each

If H₁/H₂ hold, valuation calibrates decade-ahead expectations. H₃ keeps us honest: it's an expectation tool, not a market-timing rule.

## 3 · How we'd know — the protocol

corr & R² of CAPE vs fwd-10y → valuation buckets → implied-return fit → the same test at 1 year.

## 4 · The teardown

### 4.1 The forecasting power, by horizon

In [2]:
print('10-year:', {k:round(v,3) for k,v in ten.items()})
print(' 1-year:', {k:round(v,3) for k,v in one.items()})

10-year: {'corr_cape': -0.51, 'corr_yield': 0.532, 'r2': 0.283, 'n': 1397}
 1-year: {'corr_cape': -0.195, 'corr_yield': 0.219, 'r2': 0.048, 'n': 1485}


> 💡 *In plain words:* 10y R² 0.28 (strong), 1y R² 0.05 (weak). **H₁ and H₃ hold.**

### 4.2 The valuation ladder

In [3]:
display((st.bucket_returns(d['cape'],d['fwd10'],n=5)*100).round(1))

bucket
q1    11.0
q2     8.1
q3     5.7
q4     5.1
q5     3.3
Name: fwd, dtype: float64

> 💡 *In plain words:* monotone from cheap to expensive. **H₂ holds.**

### 4.3 The implied-return fit (a concrete forecast)

In [4]:
b0,b1=st.implied_return(d['cape'],d['fwd10'])
cur=d['cape'].dropna().iloc[-1]
print(f'fwd10 = {b0:+.3f} + {b1:.2f}/CAPE')
print(f'at CAPE {cur:.0f} -> implied 10y real return {b0+b1/cur:+.1%}')

fwd10 = +0.002 + 0.90/CAPE
at CAPE 31 -> implied 10y real return +3.1%


> 💡 *In plain words:* today's rich CAPE implies a modest ~3%/yr real decade — sober expectations, not a crash call.

## 5 · The verdict

H₁, H₂ hold; H₃ confirms the limit → Signal `REAL`, Tradability `FRAGILE`, long-run forecasting `CONFIRMED`.

## 6 · Could you trade it?

As an allocation/expectations input, yes; as a market timer, no (1y R² 0.05). 'Sin a little' on valuation (Asness et al. 2017) — tilt slowly, don't switch.

## 7 · Going further

Forks: (a) excess CAPE yield (CAPE earnings yield minus real bond yield) — the *honest* version of the Fed Model; (b) cross-country CAPE (the PWB entry); (c) CAPE-tilted allocation vs constant-weight. Backlog: [`docs/pwb_strategies_inventory.md`](../../../docs/pwb_strategies_inventory.md).